In [ ]:
#!/usr/bin/env python3

import csv
import json
import os
import time
from datetime import datetime, timezone, timedelta

import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "all_59k_accidents_raw1.csv"
OUTPUT_JSONL = "open_meteo_historical_forecast_1.jsonl"
ERRORS_CSV = "open_meteo_errors.csv"

REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0
SLEEP_BETWEEN_REQUESTS = 0.0

MONTHLY_CALL_BUDGET = int(os.getenv("OPEN_METEO_CALL_BUDGET", "2000000"))

OPEN_METEO_API_KEY = os.getenv("OPEN_METEO_", "").strip()

# ✔ Historical forecast API (IMPORTANT)
FORECAST_ENDPOINT = "https://historical-forecast-api.open-meteo.com/v1/forecast"

TIMEZONE = "UTC"
TIMEFORMAT = "iso8601"
CELL_SELECTION = "land"

BATCH_SIZE = 10

session = requests.Session()
calls_made = 0


# ============================================================
# VARIABLES
# ============================================================
FORECAST_HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation_probability",
    "precipitation",
    "rain",
    "showers",
    "snowfall",
    "snow_depth",
    "weather_code",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "visibility",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]


# ============================================================
# HELPERS
# ============================================================

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def trim_to_accident_time(payload, accident_dt):
    if "hourly" not in payload:
        return payload

    times = payload["hourly"].get("time", [])
    if not times:
        return payload

    filtered = {k: [] for k in payload["hourly"].keys()}

    for i, t in enumerate(times):
        dt = datetime.fromisoformat(t.replace("Z", "+00:00")).astimezone(timezone.utc)
        if dt <= accident_dt:
            for k in filtered:
                filtered[k].append(payload["hourly"][k][i])

    payload["hourly"] = filtered
    return payload


def parse_accident_datetime_utc(row):
    raw_date = str(row.get("accidentdate", "")).strip()
    raw_time = str(row.get("time", "")).strip()

    date_part = datetime.strptime(raw_date, "%d-%b-%y")
    time_part = datetime.strptime(raw_time, "%H:%M")

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,
    )


# ============================================================
# MODEL CYCLE ALIGNMENT (CORRECT VERSION)
# ============================================================

def select_model_run(reference_dt):
    cycle_hours = [0, 6, 12, 18]
    candidate = None

    for h in cycle_hours:
        run = reference_dt.replace(hour=h, minute=0, second=0, microsecond=0)
        if run <= reference_dt:
            candidate = run

    if candidate is None:
        candidate = (reference_dt - timedelta(days=1)).replace(
            hour=18, minute=0, second=0, microsecond=0
        )

    return candidate


# ============================================================
# REQUEST LOGIC
# ============================================================

def request_json(endpoint, params):
    global calls_made

    attempt = 0
    wait = 2.0

    while attempt < MAX_RETRIES:
        try:
            calls_made += 1
            resp = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)

            if resp.status_code == 429:
                time.sleep(60)
                attempt += 1
                continue

            resp.raise_for_status()
            return True, resp.json()

        except Exception:
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, {"error": "max retries"}


def merge_payloads(target, source):
    if not target:
        target = {}

    for k, v in source.items():
        if k not in target:
            target[k] = v
        elif isinstance(v, dict):
            target[k].update(v)

    return target


def fetch_hourly(base_params):
    params = dict(base_params)
    params["hourly"] = ",".join(FORECAST_HOURLY_VARS)

    ok, data = request_json(FORECAST_ENDPOINT, params)
    return ok, data, 1


# ============================================================
# CORE FORECAST FETCH (HISTORICAL FORECAST)
# ============================================================

def fetch_forecast_for_row(row_idx, lat, lon, accident_dt):
    target_dt = accident_dt - timedelta(hours=24)
    model_run_time = select_model_run(target_dt)

    base_params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "timezone": TIMEZONE,
        "timeformat": TIMEFORMAT,
        "cell_selection": CELL_SELECTION,
    }

    if OPEN_METEO_API_KEY:
        base_params["apikey"] = OPEN_METEO_API_KEY

    base_params.update({
        "start_date": model_run_time.date().isoformat(),
        "end_date": accident_dt.date().isoformat(),
    })

    ok, payload, reqs = fetch_hourly(base_params)
    payload = trim_to_accident_hour(payload, accident_dt)

    meta = {
        "endpoint": FORECAST_ENDPOINT,
        "requests": reqs,
        "accident_time_utc": accident_dt.isoformat(),
        "target_time_utc": target_dt.isoformat(),
        "model_run_time_utc": model_run_time.isoformat(),
        "model_cycle_hour": model_run_time.hour,
    }

    return ok, meta, payload

def trim_to_accident_hour(payload, accident_dt):
    if "hourly" not in payload:
        return payload

    times = payload["hourly"].get("time", [])
    if not times:
        return payload

    target_hour = accident_dt.replace(
        minute=0,
        second=0,
        microsecond=0
    )

    chosen_idx = None

    for i, t in enumerate(times):
        dt = datetime.fromisoformat(t.replace("Z", "+00:00"))

        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)

        if dt == target_hour:
            chosen_idx = i
            break

    if chosen_idx is None:
        return payload

    payload["hourly"] = {
        k: [v[chosen_idx]]
        for k, v in payload["hourly"].items()
    }

    return payload


# ============================================================
# MAIN LOOP
# ============================================================

def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]

    if os.path.exists(ERRORS_CSV):
        os.remove(ERRORS_CSV)

    error_header = False

    success = 0
    errors = 0

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        if calls_made >= MONTHLY_CALL_BUDGET:
            break

        try:
            row = row.fillna("")
            lat = float(row.get("latitude", row.get("source_latitude")))
            lon = float(row.get("longitude", row.get("source_longitude")))

            accident_dt = parse_accident_datetime_utc(row)
            if accident_dt.year < 2022:
                continue

            ok, meta, forecast = fetch_forecast_for_row(
                idx, lat, lon, accident_dt
            )

            out = {
                "_row_index": idx,
                "accident_datetime_utc": accident_dt.isoformat(),
                "source_latitude": lat,
                "source_longitude": lon,

                "forecast_status": "ok" if ok else "partial",
                "forecast_request_count": meta["requests"],
                "forecast": forecast,

                "model_run_time_utc": meta["model_run_time_utc"],
                "model_cycle_hour": meta["model_cycle_hour"],
            }

            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(out) + "\n")

            success += 1

        except Exception as e:
            errors += 1

            with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                if not error_header:
                    w.writerow(["row_index", "error"])
                    error_header = True
                w.writerow([idx, str(e)])

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"Done. success={success}, errors={errors}, calls={calls_made}")


if __name__ == "__main__":
    run_fetcher()

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================

file1_path = "/content/all_59k_accidents_archive_weather_enriched_final_no_nan.csv"
file2_path = "/content/Stats19_TfL_merge_poi.csv"

# FINAL OUTPUT
output_path = "/content/all_59k_accidents_archive_weather_poi.csv"

# OPTIONAL LOG FILES
matched_output_path = "/content/matched_rows.csv"
unmatched_output_path = "/content/unmatched_rows.csv"

# =========================================================
# LOAD FILES
# =========================================================

print("Loading CSV files...")

df1 = pd.read_csv(file1_path, low_memory=False)
df2 = pd.read_csv(file2_path, encoding='latin1', low_memory=False)

print(f"File1 rows: {len(df1):,}")
print(f"File2 rows: {len(df2):,}")

# =========================================================
# VALIDATE REQUIRED COLUMNS
# =========================================================

required_cols = ["latitude", "longitude"]

for col in required_cols:
    if col not in df1.columns:
        raise ValueError(f"{col} missing from file1")

    if col not in df2.columns:
        raise ValueError(f"{col} missing from file2")

# =========================================================
# FORCE NUMERIC TYPES
# (NO ROUNDING)
# =========================================================

for df in [df1, df2]:
    df["latitude"] = pd.to_numeric(
        df["latitude"],
        errors="coerce"
    )

    df["longitude"] = pd.to_numeric(
        df["longitude"],
        errors="coerce"
    )

# =========================================================
# SELECT COLUMNS FROM FILE2 TO APPEND TO FILE1
# Exclude merge keys
# =========================================================

wind_idx = df2.columns.get_loc("vehicle_manoeuvres_simple")
extra_cols = list(df2.columns[wind_idx + 1:])

print(f"\nColumns being appended from file2 ({len(extra_cols)}):")
print(extra_cols)

# =========================================================
# MERGE
# KEEP ALL ROWS FROM FILE1
# APPEND MATCHING DATA FROM FILE2
# =========================================================

print("\nMerging files...")

if len(df1) != len(df2):
    raise ValueError(f"Row count mismatch: file1={len(df1)}, file2={len(df2)}")

coord_match = df1[["latitude", "longitude"]].reset_index(drop=True).equals(
    df2[["latitude", "longitude"]].reset_index(drop=True)
)

if not coord_match:
    mismatch = df1[["latitude", "longitude"]].reset_index(drop=True).ne(
        df2[["latitude", "longitude"]].reset_index(drop=True)
    ).any(axis=1)
    bad_rows = mismatch[mismatch].index.tolist()[:20]
    raise ValueError(f"Latitude/longitude mismatch at rows: {bad_rows}")

merged = pd.concat(
    [
        df1.reset_index(drop=True),
        df2[extra_cols].reset_index(drop=True)
    ],
    axis=1
)

# =========================================================
# MATCH LOGGING
# =========================================================

matched_count = len(merged)
unmatched_count = 0
total_rows = len(merged)
match_rate = 100.0

unmatched = pd.DataFrame(columns=["latitude", "longitude"])

# =========================================================
# SAVE LOG FILES
# =========================================================

print("\nSaving log files...")

merged.to_csv(matched_output_path, index=False)
unmatched.to_csv(unmatched_output_path, index=False)

print(f"Matched rows log:   {matched_output_path}")
print(f"Unmatched rows log: {unmatched_output_path}")

# =========================================================
# SAVE FINAL FILE
# =========================================================

merged.to_csv(output_path, index=False)

print(f"\nFinal merged file saved:")
print(output_path)

print("\nDONE.")

In [ ]:
import csv

# Input and output file names
input_file = "Stats19_TfL_merge_poi.csv"
output_file = "output.csv"

# Read the first row (headers) and first data row
with open(input_file, mode="r", newline="", encoding="utf-8") as infile:
    reader = csv.reader(infile)

    headers = next(reader)      # First row = headers
    first_row = next(reader)    # Second row = first data row

# Write headers + first row to new CSV
with open(output_file, mode="w", newline="", encoding="utf-8") as outfile:
    writer = csv.writer(outfile)

    writer.writerow(headers)
    writer.writerow(first_row)

print(f"Created {output_file} with headers and first row.")

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================


output_path = "/content/all_59k_accidents_archive_weather_enriched_final.csv"

df1 = pd.read_csv(output_path, low_memory=False)

print(f"File1 rows: {len(df1):,}")

In [ ]:
import pandas as pd

# Input/output file paths
input_csv = "all_59k_accidents_archive_weather_poi.csv"
output_csv = "all_59k_accidents_archive_weather_poi_final.csv"

# Columns to drop
cols_to_drop = [
    'nearest_pub_road_fhrsid',
    'nearest_pub_road_name',
    'nearest_school_road_name',
    'nearest_overground_road_id',
    'nearest_overground_road_name',
    'nearest_underground_road_id',
    'nearest_underground_road_name',
    'nearest_signal_road_id',
    'nearest_signal_road_name',
    'nearest_bus_stop_road_id',
    'Highstreet_name',
]

# Load CSV
df = pd.read_csv(input_csv)

# Drop columns (ignore missing columns)
df = df.drop(columns=cols_to_drop, errors='ignore')

# Save cleaned CSV
df.to_csv(output_csv, index=False)

print(f"Saved cleaned file to: {output_csv}")